# Nested Monte Carlo Power Simulator for the Degradation Cluster-Permutation Test

Mirrors `pilot_noise_cluster_permutation_power.ipynb`, adapted for degradation
severity instead of SNR. Same reasoning: cluster-permutation has no closed-form
power formula, so this simulates synthetic pilot datasets under a ground truth
fit from the (corrected, region-as-subject) degradation pilot, runs the actual
cluster-permutation procedure on each one, and reports the hit rate as power
across a grid of candidate `(k regions, R repetitions, D connectome draws,
severity levels)` designs.

**Differences from the noise-sweep version**:
- Only one comparison (`wCSET` vs `CSET` -- study 2 was scoped to 2 methods from
  the start), not five.
- Severity is a **linear** percentage scale (0-100%), not log-SNR -- interpolation
  for candidate denser severity grids uses linear spacing, not log spacing.
- **Adds a connectome-draw dimension** the first version of this notebook
  omitted entirely: `pilot_degradation_02_solve.py` (the data this notebook's
  ground truth comes from) uses exactly one representative corrupted connectome
  per severity level, so on its own it carries zero information about
  draw-to-draw variability. Draw noise is structurally different from region or
  repetition noise: a single corrupted connectome is reconstructed against
  *every* region at that severity, so its idiosyncrasy is a **shared offset
  across all regions for a given draw**, not an independent per-region or
  per-repetition term. Modeled that way below, using `sigma2_method_draw`
  already fit by `pilot_degradation_draws_power_analysis.ipynb` (essentially 0
  for AP/correlation, real but modest for localization error, 0.01-6.1 across
  severities).

**Ground truth from the corrected pilot data**: per outcome, the 26-region x
6-severity paired-difference matrix (region-as-subject, averaged over the 8 matched
repetitions -- see `pilot_degradation_cluster_permutation.ipynb`, fixed after an
earlier version incorrectly treated `(region, repetition)` pairs as 208 independent
subjects) gives `true_mean_diff(severity)`, `sigma2_region_diff(severity)` (via
method-of-moments, same as the noise version), and `sigma2_resid(severity)` (reused
directly from `pilot_degradation_variance_components.json`, no refitting).
`sigma2_method_draw(severity)` is reused directly from
`pilot_degradation_draws_variance_components.json` (0 at severity=0 by
construction -- no corruption means no draw-to-draw variability to begin with).

**Written to run later, not now** -- iteration counts below are smoke-test sized.

In [1]:
import json
import numpy as np
import pandas as pd
from scipy import stats

OUTCOMES = ["localization_error_hrp", "ap_hrp", "ap_pp", "correlation_auc"]
HIGHER_IS_BETTER = {"localization_error_hrp": False, "ap_hrp": True, "ap_pp": True, "correlation_auc": True}
REFERENCE_METHOD = "wcset"
COMPETITOR = "cset"
SEVERITY_EXISTING = [0, 20, 40, 60, 80, 100]
R_PILOT = 8   # repetitions actually collected per (region, severity) in pilot_degradation_02_solve.py

df = pd.read_csv("pilot_degradation_outcomes.tsv", sep="\t")
vc = json.load(open("pilot_degradation_variance_components.json"))
vc_draws = json.load(open("pilot_degradation_draws_variance_components.json"))
regions = sorted(df["region"].unique())
k_pilot = len(regions)
print(f"{len(df)} rows, {k_pilot} regions (pilot), existing severities {SEVERITY_EXISTING}")

2496 rows, 26 regions (pilot), existing severities [0, 20, 40, 60, 80, 100]


## Ground truth per outcome: mean difference + region variance vs. severity

Fit once from the corrected pilot data, then interpolated (linearly in severity,
not log) to whatever candidate severity grid a given design point uses.

In [2]:
def region_series(outcome):
    """(k_pilot, 6) matrix: per-region, per-severity mean paired difference
    (wcset - cset), sign-adjusted so positive = wcset better, averaged over the 8
    matched repetitions."""
    sign = 1.0 if HIGHER_IS_BETTER[outcome] else -1.0
    wide_wcset = df[df["method"] == REFERENCE_METHOD].groupby(["region", "percentage"])[outcome].mean().unstack()[SEVERITY_EXISTING]
    wide_comp = df[df["method"] == COMPETITOR].groupby(["region", "percentage"])[outcome].mean().unstack()[SEVERITY_EXISTING]
    diff = sign * (wide_wcset - wide_comp)
    return diff.reindex(regions).to_numpy()


def fit_ground_truth(outcome):
    D = region_series(outcome)
    sigma2_resid_existing = np.array([vc[str(s)][outcome]["sigma2_resid"] for s in SEVERITY_EXISTING])
    noise_floor = 2 * sigma2_resid_existing / R_PILOT
    observed_var = D.var(axis=0, ddof=1)
    sigma2_region_diff_existing = np.maximum(0.0, observed_var - noise_floor)
    # sigma2_method_draw: 0 at severity=0 by construction (no corruption, no draw-to-draw variability)
    sigma2_draw_existing = np.array([
        0.0 if s == 0 else vc_draws[str(s)][outcome]["sigma2_method_draw"] for s in SEVERITY_EXISTING
    ])
    return {
        "severity_existing": np.array(SEVERITY_EXISTING, dtype=float),
        "true_mean_diff_existing": D.mean(axis=0),
        "sigma2_region_diff_existing": sigma2_region_diff_existing,
        "sigma2_resid_existing": sigma2_resid_existing,
        "sigma2_draw_existing": sigma2_draw_existing,
    }


def interp_ground_truth(gt, severity_values):
    x = np.array(severity_values, dtype=float)
    xp = gt["severity_existing"]   # already increasing (0->100), no flip needed (unlike log-snr)
    true_mean = np.interp(x, xp, gt["true_mean_diff_existing"])
    sigma2_region = np.interp(x, xp, gt["sigma2_region_diff_existing"])
    sigma2_resid = np.interp(x, xp, gt["sigma2_resid_existing"])
    sigma2_draw = np.interp(x, xp, gt["sigma2_draw_existing"])
    return true_mean, sigma2_region, sigma2_resid, sigma2_draw


ground_truths = {o: fit_ground_truth(o) for o in OUTCOMES}
print("Ground truth fit for", len(ground_truths), "outcomes.")

Ground truth fit for 4 outcomes.


## Cluster-permutation test (region-as-subject, unchanged from the companion notebook)

In [3]:
def cluster_permutation_test(D, cluster_alpha=0.05, n_perm=500, seed=0):
    """D: (k, S) array, one series per region (subject). Region-level permutation:
    flips each row's sign consistently across all S columns."""
    k, S = D.shape
    rng = np.random.default_rng(seed)
    t_crit = stats.t.ppf(1 - cluster_alpha / 2, df=k - 1)

    def compute_t(Dperm):
        m = Dperm.mean(axis=0)
        se = Dperm.std(axis=0, ddof=1) / np.sqrt(k)
        return m / se

    def find_clusters(t):
        clusters = []
        sign_mask = np.where(t > t_crit, 1, np.where(t < -t_crit, -1, 0))
        i = 0
        while i < S:
            if sign_mask[i] != 0:
                j = i
                while j < S and sign_mask[j] == sign_mask[i]:
                    j += 1
                clusters.append((i, j - 1, np.sum(t[i:j]), sign_mask[i]))
                i = j
            else:
                i += 1
        return clusters

    t_obs = compute_t(D)
    clusters_obs = find_clusters(t_obs)

    max_pos_null = np.zeros(n_perm)
    for p in range(n_perm):
        flips = rng.choice([-1.0, 1.0], size=k)
        t_p = compute_t(D * flips[:, None])
        pos = [c[2] for c in find_clusters(t_p) if c[3] == 1]
        max_pos_null[p] = max(pos) if pos else 0.0

    results = []
    for (i, j, mass, sgn) in clusters_obs:
        if sgn == 1:
            pval = (np.sum(max_pos_null >= mass) + 1) / (n_perm + 1)
            results.append({"i": i, "j": j, "mass": mass, "p_value": pval})
    return results

## Simulate one synthetic pilot dataset for a candidate (k, R, D, severity_values) design

`draw_effect` is deliberately **one shared value per severity level per outer
iteration**, not per region -- a specific corrupted connectome's idiosyncrasy
applies identically to every region reconstructed against it. Averaging `D`
independent draws shrinks its variance the normal `1/D` way, same mechanism as
`R` shrinking `sigma2_resid`, just applied at a different level of the model.

In [4]:
def simulate_region_series(true_mean, sigma2_region, sigma2_resid, sigma2_draw, k, R, D, rng):
    """(k, S) synthetic paired-difference matrix."""
    S = len(true_mean)
    z_region = rng.normal(0, 1, size=k)
    region_effect = z_region[:, None] * np.sqrt(sigma2_region)[None, :]
    draw_effect = rng.normal(0, 1, size=S) * np.sqrt(sigma2_draw / D)   # shared across regions, not per-region
    resid_noise = rng.normal(0, 1, size=(k, S)) * np.sqrt(2 * sigma2_resid / R)[None, :]
    return true_mean[None, :] + region_effect + draw_effect[None, :] + resid_noise


def true_cluster_mask(true_mean, min_effect_frac=0.1):
    return true_mean >= min_effect_frac * true_mean.max()

## Nested power sweep

**Cost warning**: same as the noise-sweep version -- values below are smoke-test
sized. Restore `N_OUTER=300`, `INNER_N_PERM=500`, and a fuller `DESIGN_GRID` /
`OUTCOMES_TO_TEST` for the real run. Localization error is the outcome most worth
spending compute on for both the region and draw dimensions -- it's the only one
with real `sigma2_method_region` *and* real `sigma2_method_draw` structure;
AP/correlation are already saturated on both.

A first pass at `D` (draw count), `k=26, R=8` fixed, `N_OUTER=150`: power stayed
at 0.93-0.97 across `D=1,3,5,8` for localization error -- i.e. **even one draw
per severity already gives adequate power for "is there an average wCSET
advantage."** That's a different, easier question than the categorical
`method:draw` interaction test in `pilot_degradation_draws_power_analysis.ipynb`
(never cleared its own grid) asks -- that one tests whether draw identity
changes *which region-level pattern* you'd see, not just whether an average
effect exists. Both answers are legitimate; which one matters depends on
whether the real study needs to defend against "you got lucky with your
connectome draw" specifically, or just needs the average-effect claim.

In [5]:
N_OUTER = 20             # smoke-test default -- raise to ~200-300 for the real run
INNER_N_PERM = 100        # smoke-test default -- raise to ~500 for the real run
SEED = 0

DESIGN_GRID = [
    {"k": 26, "R": 8, "D": 1, "severity_values": [0, 20, 40, 60, 80, 100]},   # the already-run pilot 2a (D=1), as a baseline
    {"k": 26, "R": 8, "D": 5, "severity_values": [0, 20, 40, 60, 80, 100]},   # pilot 2b's own draw count
    {"k": 20, "R": 4, "D": 3, "severity_values": [0, 20, 40, 60, 80, 100]},
]

OUTCOMES_TO_TEST = ["localization_error_hrp"]   # smoke test -- expand to all 4 for the real run

rng = np.random.default_rng(SEED)
power_rows = []

for design in DESIGN_GRID:
    k, R, Dn, severity_values = design["k"], design["R"], design["D"], design["severity_values"]
    for outcome in OUTCOMES_TO_TEST:
        gt = ground_truths[outcome]
        true_mean, sigma2_region, sigma2_resid, sigma2_draw = interp_ground_truth(gt, severity_values)
        true_mask = true_cluster_mask(true_mean)

        hits = 0
        for _ in range(N_OUTER):
            D = simulate_region_series(true_mean, sigma2_region, sigma2_resid, sigma2_draw, k, R, Dn, rng)
            clusters = cluster_permutation_test(D, n_perm=INNER_N_PERM, seed=rng.integers(1e9))
            hit = any(true_mask[c["i"]:c["j"] + 1].any() and c["p_value"] < 0.05 for c in clusters)
            hits += hit

        power_rows.append({
            "k": k, "R": R, "D_draws": Dn, "S": len(severity_values), "outcome": outcome, "power": hits / N_OUTER,
        })
        print(f"k={k}, R={R}, D={Dn}, S={len(severity_values)}, {outcome}: power={hits/N_OUTER:.2f}")

power_df = pd.DataFrame(power_rows)
power_df

k=26, R=8, D=1, S=6, localization_error_hrp: power=0.85
k=26, R=8, D=5, S=6, localization_error_hrp: power=0.95
k=20, R=4, D=3, S=6, localization_error_hrp: power=0.75


,k,R,D_draws,S,outcome,power
0,26,8,1,6,localization_error_hrp,0.85
1,26,8,5,6,localization_error_hrp,0.95
2,20,4,3,6,localization_error_hrp,0.75


## Running the real sweep

Restore full iteration counts, expand `OUTCOMES_TO_TEST`, and widen `DESIGN_GRID`
with more `(k, R, severity_values)` combinations -- including denser severity
grids (e.g. every 10% instead of every 20%) if finer localization of where wCSET's
advantage starts/stops matters for the writeup -- before running this for real.